# Credexa AI — EfficientNet-B4 Tamper Detection Training

Trains on **CASIA v1 + v2** (14,281 images) for binary classification:
- **Class 0:** Authentic
- **Class 1:** Tampered / Spliced

Saves `efficientnet_b4_tamper.pth` — drop this into `models/trained/efficientnet_b4_tamper/` in your Credexa project.

In [ ]:
# @title 1. Mount Google Drive & Install deps
from google.colab import drive
drive.mount('/content/drive')

# Install torchvision with CUDA support (already on Colab)
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title 2. Upload or symlink the CASIA dataset
import os
from pathlib import Path

# Two options:
# Option A: Upload casia_combined folder to Drive and symlink here
# Option B: Upload directly to Colab runtime (ephemeral)

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

# Try linking from Drive first
drive_casia = Path('/content/drive/MyDrive/casia_combined')
if drive_casia.exists():
    !ln -sf /content/drive/MyDrive/casia_combined /content/data/casia_combined
    print("Linked casia_combined from Google Drive")
else:
    print("NOTE: casia_combined not found in Drive.")
    print("Upload casia_combined to /content/drive/MyDrive/casia_combined or use Option B below.")
    print()
    print("Option B — Upload via Colab file panel:")
    from google.colab import files
    uploaded = files.upload()
    # Assumes a zip file named casia_combined.zip
    import zipfile
    with zipfile.ZipFile('casia_combined.zip', 'r') as zf:
        zf.extractall('/content/data/')
    print("Extracted casia_combined")

# Verify
casia_path = DATA_DIR / 'casia_combined'
if casia_path.exists():
    train_au = len(list((casia_path / 'train' / 'Au').iterdir()))
    train_tp = len(list((casia_path / 'train' / 'Tp').iterdir()))
    test_au = len(list((casia_path / 'test' / 'Au').iterdir()))
    test_tp = len(list((casia_path / 'test' / 'Tp').iterdir()))
    print(f"\nDataset verified:")
    print(f"  Train: {train_au} authentic + {train_tp} tampered = {train_au + train_tp}")
    print(f"  Test:  {test_au} authentic + {test_tp} tampered = {test_au + test_tp}")
else:
    raise FileNotFoundError("casia_combined not found. Upload the dataset first.")

In [ ]:
# @title 3. Define dataset & model
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
import time

# ─── Config ──────────────────────────────────────────────────────────
NUM_EPOCHS = 75        # @param {type:"slider", min:10, max:150, step:5}
BATCH_SIZE = 32          # @param {type:"integer"}
LEARNING_RATE = 1e-4     # @param {type:"number"}
IMG_SIZE = 380
WEIGHT_DECAY = 1e-5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ─── Data Augmentation (train) vs standard (val) ────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ─── Dataset ─────────────────────────────────────────────────────────
class TamperDataset(Dataset):
    def __init__(self, root, split='train', transform=None):
        self.samples = []
        self.transform = transform
        casia = Path(root) / 'casia_combined'
        for cls, label in [('Au', 0), ('Tp', 1)]:
            class_dir = casia / split / cls
            if class_dir.exists():
                for fname in class_dir.iterdir():
                    if fname.suffix.lower() in ('.jpg', '.png', '.tif', '.tiff'):
                        self.samples.append((str(fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_ds = TamperDataset(DATA_DIR, 'train', train_transform)
val_ds = TamperDataset(DATA_DIR, 'test', val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds)} samples ({len(val_loader)} batches)")
print(f"\nClass distribution:")
print(f"  Authentic (0): {sum(1 for _, l in train_ds.samples if l==0)} train / {sum(1 for _, l in val_ds.samples if l==0)} val")
print(f"  Tampered  (1): {sum(1 for _, l in train_ds.samples if l==1)} train / {sum(1 for _, l in val_ds.samples if l==1)} val")

In [ ]:
# @title 4. Build model
model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Cosine annealing LR schedule
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Model: EfficientNet-B4")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# @title 5. Train
from tqdm.notebook import tqdm

MODEL_SAVE_DIR = Path('/content/drive/MyDrive/credexa_models')
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

best_acc = 0.0
train_losses = []
val_accs = []

for epoch in range(NUM_EPOCHS):
    # ─── Train ────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ─── Validate ─────────────────────────────────────────────────────
    model.eval()
    correct = total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [val]', leave=False):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    acc = correct / total
    val_accs.append(acc)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | loss={avg_loss:.4f} | val_acc={acc:.4f} | lr={current_lr:.2e}')

    # ─── Save best ────────────────────────────────────────────────────
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), MODEL_SAVE_DIR / 'efficientnet_b4_tamper.pth')
        print(f'  >> New best model saved (acc={acc:.4f})')

    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': acc,
            'loss': avg_loss,
        }, MODEL_SAVE_DIR / f'checkpoint_epoch_{epoch+1}.pt')

print(f'\n=== Training Complete ===')
print(f'Best val_acc: {best_acc:.4f}')
print(f'Model saved to: {MODEL_SAVE_DIR / "efficientnet_b4_tamper.pth"}')

In [ ]:
# @title 6. Plot training curve
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

axes[1].plot(val_accs)
axes[1].axhline(y=best_acc, color='r', linestyle='--', label=f'Best: {best_acc:.3f}')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(MODEL_SAVE_DIR / 'training_curve.png')
plt.show()
print(f'Plot saved to {MODEL_SAVE_DIR / "training_curve.png"}')

In [ ]:
# @title 7. Evaluate: per-class metrics & confusion matrix
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import numpy as np

# Load best model
model.load_state_dict(torch.load(MODEL_SAVE_DIR / 'efficientnet_b4_tamper.pth'))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Final evaluation'):
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

print('=== Classification Report ===')
print(classification_report(all_labels, all_preds, target_names=['Authentic (0)', 'Tampered (1)']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Authentic', 'Tampered'],
            yticklabels=['Authentic', 'Tampered'])
plt.title(f'Confusion Matrix (acc={np.mean(np.array(all_labels)==np.array(all_preds)):.3f})')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(MODEL_SAVE_DIR / 'confusion_matrix.png')
plt.show()

auc = roc_auc_score(all_labels, all_probs)
print(f'\nROC-AUC: {auc:.4f}')
print(f'False Positive Rate (tampered classified as authentic): {cm[1,0]/max(sum(cm[1]),1):.3f}')
print(f'False Negative Rate (authentic classified as tampered): {cm[0,1]/max(sum(cm[0]),1):.3f}')

In [ ]:
# @title 8. Download the trained model
from google.colab import files

model_path = MODEL_SAVE_DIR / 'efficientnet_b4_tamper.pth'
print(f'Model size: {model_path.stat().st_size / 1024 / 1024:.1f} MB')
files.download(str(model_path))

# Instructions:
print('\n=== Deployment Instructions ===')
print(f'1. Download efficientnet_b4_tamper.pth above')
print(f'2. Place it in: Credexa.AI/models/trained/efficientnet_b4_tamper/')
print(f'3. Restart the Credexa backend')
print(f'4. The visual_forensics pipeline will auto-load it')